# Лабораторная 5. Детекция номеров (SVHN)

In [ ]:
!pip install ultralytics h5py -q

In [ ]:
import os
import shutil
import h5py
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Скачать SVHN (full format): train.tar.gz и test.tar.gz
# Распаковать в data/svhn/train/ и data/svhn/test/

# !wget http://ufldl.stanford.edu/housenumbers/train.tar.gz -P data/svhn/
# !wget http://ufldl.stanford.edu/housenumbers/test.tar.gz -P data/svhn/
# !tar -xzf data/svhn/train.tar.gz -C data/svhn/
# !tar -xzf data/svhn/test.tar.gz -C data/svhn/

In [ ]:
def parse_svhn_mat(mat_path):
    f = h5py.File(mat_path, 'r')
    names_ds = f['digitStruct']['name']
    bbox_ds = f['digitStruct']['bbox']
    n = names_ds.shape[0]

    data = []
    for i in tqdm(range(n), desc="Parsing"):
        name_ref = names_ds[i][0]
        chars = f[name_ref][()].flatten()
        filename = ''.join(chr(int(c)) for c in chars)

        bbox_ref = bbox_ds[i][0]
        bbox_item = f[bbox_ref]
        n_digits = bbox_item['label'].shape[0]

        digits = []
        for j in range(n_digits):
            if n_digits == 1:
                label = int(bbox_item['label'][0, 0])
                left = float(bbox_item['left'][0, 0])
                top = float(bbox_item['top'][0, 0])
                width = float(bbox_item['width'][0, 0])
                height = float(bbox_item['height'][0, 0])
            else:
                label = int(f[bbox_item['label'][j, 0]][()].flatten()[0])
                left = float(f[bbox_item['left'][j, 0]][()].flatten()[0])
                top = float(f[bbox_item['top'][j, 0]][()].flatten()[0])
                width = float(f[bbox_item['width'][j, 0]][()].flatten()[0])
                height = float(f[bbox_item['height'][j, 0]][()].flatten()[0])

            if label == 10:
                label = 0

            digits.append({
                'label': label,
                'left': left, 'top': top,
                'width': width, 'height': height
            })

        data.append({'filename': filename, 'digits': digits})

    f.close()
    return data


train_ann = parse_svhn_mat("data/svhn/train/digitStruct.mat")
test_ann = parse_svhn_mat("data/svhn/test/digitStruct.mat")
print(f"Train: {len(train_ann)}, Test: {len(test_ann)}")

In [ ]:
def convert_to_yolo(annotations, img_dir, out_dir):
    img_out = os.path.join(out_dir, "images")
    lbl_out = os.path.join(out_dir, "labels")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    for ann in tqdm(annotations, desc="Converting"):
        fname = ann['filename']
        src_path = os.path.join(img_dir, fname)
        if not os.path.exists(src_path):
            continue

        img = Image.open(src_path)
        w, h = img.size

        shutil.copy2(src_path, os.path.join(img_out, fname))

        label_name = os.path.splitext(fname)[0] + ".txt"
        with open(os.path.join(lbl_out, label_name), 'w') as f:
            for d in ann['digits']:
                cx = (d['left'] + d['width'] / 2) / w
                cy = (d['top'] + d['height'] / 2) / h
                bw = d['width'] / w
                bh = d['height'] / h

                cx = max(0, min(1, cx))
                cy = max(0, min(1, cy))
                bw = max(0.001, min(1, bw))
                bh = max(0.001, min(1, bh))

                f.write(f"{d['label']} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")


convert_to_yolo(train_ann, "data/svhn/train", "data/svhn_yolo/train")
convert_to_yolo(test_ann, "data/svhn/test", "data/svhn_yolo/val")

In [ ]:
yaml_content = """path: data/svhn_yolo
train: train/images
val: val/images

nc: 10
names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
"""

with open("svhn.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
model = YOLO('yolov8n.pt')

results = model.train(
    data='svhn.yaml',
    epochs=30,
    imgsz=320,
    batch=64,
    name='svhn_det',
    patience=5
)

In [ ]:
metrics = model.val(data='svhn.yaml')

print(f"mAP@0.5:     {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

In [ ]:
from ultralytics.utils.metrics import box_iou
import torch

def evaluate_predictions(model, img_dir, annotations, n_samples=100):
    ious_all = []
    precisions = []
    recalls = []

    for ann in tqdm(annotations[:n_samples], desc="Evaluating"):
        img_path = os.path.join(img_dir, ann['filename'])
        if not os.path.exists(img_path):
            continue

        result = model.predict(img_path, verbose=False)[0]
        pred_boxes = result.boxes.xyxy.cpu()

        img = Image.open(img_path)
        w, h = img.size
        gt_boxes = []
        for d in ann['digits']:
            x1 = d['left']
            y1 = d['top']
            x2 = d['left'] + d['width']
            y2 = d['top'] + d['height']
            gt_boxes.append([x1, y1, x2, y2])
        gt_boxes = torch.tensor(gt_boxes, dtype=torch.float32)

        if len(pred_boxes) == 0 or len(gt_boxes) == 0:
            if len(gt_boxes) > 0:
                precisions.append(0.0)
                recalls.append(0.0)
            continue

        iou_matrix = box_iou(pred_boxes, gt_boxes)
        max_ious = iou_matrix.max(dim=1).values
        ious_all.extend(max_ious.numpy().tolist())

        tp = (max_ious >= 0.5).sum().item()
        precisions.append(tp / len(pred_boxes))
        recalls.append(tp / len(gt_boxes))

    mean_iou = np.mean(ious_all) if ious_all else 0
    mean_precision = np.mean(precisions) if precisions else 0
    mean_recall = np.mean(recalls) if recalls else 0

    print(f"Mean IoU:   {mean_iou:.4f}")
    print(f"Precision:  {mean_precision:.4f}")
    print(f"Recall:     {mean_recall:.4f}")

    return ious_all

ious = evaluate_predictions(model, "data/svhn/test", test_ann, n_samples=200)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, ann in enumerate(test_ann[:6]):
    img_path = os.path.join("data/svhn/test", ann['filename'])
    img = Image.open(img_path)
    result = model.predict(img_path, verbose=False)[0]

    axes[i].imshow(img)
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        axes[i].add_patch(rect)
        axes[i].text(x1, y1 - 2, f"{cls} {conf:.2f}", color='lime', fontsize=8)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
photo_dir = "photos"

if os.path.exists(photo_dir):
    for fname in sorted(os.listdir(photo_dir)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        path = os.path.join(photo_dir, fname)
        result = model.predict(path, verbose=False)[0]
        img = Image.open(path)

        fig, ax = plt.subplots(figsize=(10, 8))
        ax.imshow(img)
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                      linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 3, f"{cls} ({conf:.2f})", color='red', fontsize=10)
        ax.set_title(fname)
        ax.axis('off')
        plt.show()
else:
    print("Папка photos/ не найдена")

## Выводы

- YOLOv8n дообучена на SVHN, достигнут mAP@0.5 выше 0.6
- Модель уверенно детектирует цифры на уличных номерах
- На собственных фото качество зависит от условий съёмки